# FINN → region emissions — Python walkthrough

This notebook runs the whole pipeline using the `finn_sc` package for the
region defined in `config/config.yml` (**default: Santa Catarina**; point
`region.boundaries_shp` at any shapefile to change it).
Run it from the project root (the folder above `notebooks/`), or adjust
the path in the first cell. See `README.md`, `docs/methodology.md` and
`docs/data_dictionary.md` for the details and **verified references**
(`REFERENCES.md`).

> **Units:** raw FINN gases are mol/day and aerosols kg/day; after step 01
> everything is in **metric tonnes**.

## ▶️ Setup — Google Colab (run these first; they auto-skip when you run locally)

**On Google Colab:** just run every cell from the top. The next three cells
upload the project zip, install the libraries, and place your shapefile —
then the pipeline cells below run unchanged.

**Locally (VS Code / Jupyter):** these three cells detect that the project is
already on disk and do nothing, so you can also just run top-to-bottom.


In [ ]:
# --- Locate / fetch the project (Colab-aware, harmless locally) --------------
import sys, os
IN_COLAB = 'google.colab' in sys.modules
def _has_project(p='.'): return os.path.exists(os.path.join(p, 'config', 'config.yml'))

# If opened from inside notebooks/, step up to the project root.
if os.path.basename(os.getcwd()) == 'notebooks' and _has_project('..'):
    os.chdir('..')

# On Colab, upload + unzip the project if it isn't here yet.
if IN_COLAB and not _has_project() and not _has_project('FINN_SC'):
    import zipfile
    from google.colab import files
    print('Choose FINN_SC.zip  (or FINN_SC-starter.zip):')
    up = files.upload()                       # pick the zip from your computer
    zname = max(up, key=lambda n: len(up[n]))  # the uploaded zip
    with zipfile.ZipFile(zname) as z:
        z.extractall('.')

# Enter the extracted folder if we are one level above it.
if not _has_project() and _has_project('FINN_SC'):
    os.chdir('FINN_SC')

sys.path.insert(0, os.path.join(os.getcwd(), 'python'))
print('Working dir:', os.getcwd(), '| project found:', _has_project())

### Install dependencies

On Colab this installs the exact libraries into the kernel. Locally it is
skipped, so pip never fights an existing conda `finn` environment.


In [ ]:
# Install requirements on Colab (skipped locally to protect a conda env).
if IN_COLAB:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=False)
    print('dependencies installed')
else:
    print('Local run - using the existing environment (e.g. the conda `finn` env).')

### Boundary shapefile

The shipped zip **already includes** the Santa Catarina sample in
`data/inputs/`, so this normally does nothing. It only prompts for an upload
when `data/inputs/` has no `.shp` yet (e.g. you are mapping a different region).


In [ ]:
# Ensure a boundary shapefile is present (the bundle already ships one).
os.makedirs('data/inputs', exist_ok=True)
_has_shp = any(f.lower().endswith('.shp') for f in os.listdir('data/inputs'))
if IN_COLAB and not _has_shp:
    import shutil
    from google.colab import files
    print('Upload the shapefile parts TOGETHER: .shp .dbf .shx .prj')
    up = files.upload()
    for name in up:
        shutil.move(name, os.path.join('data/inputs', name))
    _has_shp = any(f.lower().endswith('.shp') for f in os.listdir('data/inputs'))
print('shapefile in data/inputs:', _has_shp)

## 0. Setup — make `finn_sc` importable and load the config

In [ ]:
import sys, os
from pathlib import Path

# Work from the project root so config-relative paths resolve.
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'python'))

from finn_sc import (load_config, build_finn_urls, filter_finn_urls,
                     filter_finn_by_date, download_finn_files, convert_finn_units,
                     clip_to_region, aggregate_by_region,  # 2026-10-03 review
                     date_bounds, selected_years, filter_rows_by_date)
from finn_sc.config import resolve
import pandas as pd   # geopandas is imported later, only for steps 3-4

cfg = load_config()
sel = cfg['selection']
bounds = date_bounds(cfg)          # None unless start_date/end_date are set
years = selected_years(cfg)        # date range span, or sel['years']
print('Region:', cfg['region']['name'], '| years:', years, '| date range:', bounds)

## 1. Download FINN (step 00)

Downloads are large — this cell fetches only the configured year(s).
Set `RUN_DOWNLOAD = False` to skip if you already have the file.

In [ ]:
RUN_DOWNLOAD = True
raw_dir = resolve(cfg, cfg['paths']['raw_finn'])

if RUN_DOWNLOAD:
    all_urls = build_finn_urls()
    for year in years:
        urls = filter_finn_urls(all_urls, version=sel['version'],
                                sensor=sel['sensor'], year=year)
        if bounds: urls = filter_finn_by_date(urls, *bounds)
        download_finn_files(urls, dest_folder=raw_dir,
            combined_filename=f"{year}_finn_{sel['version']}_{sel['sensor']}.csv",
            include_source_url=cfg['download']['include_source_url'],
            min_file_size_kb=cfg['download']['min_file_size_kb'],
            timeout_seconds=cfg['download']['timeout_seconds'])

year = years[0]
raw_path = raw_dir / f"{year}_finn_{sel['version']}_{sel['sensor']}.csv"
try:
    raw = pd.read_csv(raw_path)
except pd.errors.EmptyDataError:
    raise SystemExit(
        f"{raw_path.name} is empty - no FINN files were downloaded for this "
        f"selection. Check config.selection (version must be one of v1.5 / v2.5 "
        f"/ v2.5.1 - NOT 'v2.5.1 NRT') and the dates, then re-run the cell above.")
raw.head()

## 2. Convert units to tonnes (step 01)

In [ ]:
converted = convert_finn_units(raw, cfg)
if bounds:   # keep only rows within the exact date range (adds DATE col)
    converted = filter_rows_by_date(converted, year, *bounds)
    print('rows in', bounds, '->', len(converted))
interim = resolve(cfg, cfg['paths']['interim']); interim.mkdir(parents=True, exist_ok=True)
conv_path = interim / f"{year}_finn_{sel['version']}_{sel['sensor']}_converted.csv"
converted.to_csv(conv_path, index=False)
converted[[c for c in ['CO2','CO','PM25','OC'] if c in converted.columns]].describe()

## 3. Clip to the region polygons (step 02: bbox pre-filter + point-in-polygon; 2026-10-03 review)

The bbox is auto-derived from the region shapefile (config `bbox: auto`),
so we load the shapefile now and pass it to the clip. Works for any region.

In [ ]:
import geopandas as gpd   # spatial steps need geopandas (see requirements.txt)
shp_path = resolve(cfg, cfg['region']['boundaries_shp'])
shp = gpd.read_file(shp_path) if shp_path.is_file() else None
clipped = clip_to_region(converted, cfg, shp)   # 2026-10-03 review: bbox + polygons
clip_path = interim / f"filtered_points_{year}_{sel['sensor']}_{sel['version']}_converted_within_grid.csv"
clipped.to_csv(clip_path, index=False)
print('records kept inside the region polygons:', len(clipped))
clipped.head()

## 4. Aggregate by region polygon (step 04)

Groups by `region.region_field` (default `NM_MUN`; any polygon layer works).
Needs the boundaries shapefile in `data/inputs/` (see `data/inputs/README.md`).

In [ ]:
assert shp is not None, f'Put the boundaries shapefile at {shp_path} first.'
region_field = cfg['region']['region_field']
summary = aggregate_by_region(clipped, shp, cfg,
            year=year, sensor=sel['sensor'], version=sel['version'])

out_dir = resolve(cfg, cfg['paths']['output']); out_dir.mkdir(parents=True, exist_ok=True)
final = shp.merge(summary, on=region_field, how='left')
pd.DataFrame(final.drop(columns='geometry')).to_csv(out_dir/'aggregated_emissions_all.csv', index=False)
summary.sort_values('events', ascending=False).head(10)  # 2026-10-03 review: events, not row counts

## 5. Descriptive analysis (step 05)

In [ ]:
from finn_sc.plots import (period_4panel, period_timeseries, period_summary_table,
                           annual_4panel, annual_summary_table, distinct_years)
agg = pd.read_csv(out_dir/'aggregated_emissions_all.csv')
res = cfg.get('aggregation', {}).get('time_resolution', 'year')

# Primary outputs, named after the period / resolution actually run
period_4panel(agg, sel['version'], sel['sensor'], out_dir/f'overview_{res}.png', resolution=res)
period_timeseries(agg, sel['version'], sel['sensor'], out_dir/f'timeseries_{res}.png', resolution=res)
pt = period_summary_table(agg, sel['version'], sel['sensor'], out_dir/f'period_summary_{res}.csv')
print('figure:', out_dir/f'overview_{res}.png')

# By-year rollup only when the data actually span >= 2 calendar years
if len(distinct_years(agg, sel['version'], sel['sensor'])) >= 2:
    annual_4panel(agg, sel['version'], sel['sensor'], out_dir/'overview_by_year.png')
    annual_summary_table(agg, sel['version'], sel['sensor'], out_dir/'summary_by_year.csv')
    print('by-year rollup written (overview_by_year.png, summary_by_year.csv)')
else:
    print("single-year/partial run -> by-year rollup skipped; the period outputs describe it")
pt

Top regions by PM₂.₅ (tonnes):

In [ ]:
cols = [c for c in ['PM25','CO','CO2','events'] if c in agg.columns]  # 2026-10-03 review
agg[[region_field]+cols].sort_values('PM25', ascending=False).head(10)

---
Done. Outputs are in `data/output/`. Next: style
`aggregated_emissions_all.shp` in QGIS (see the QGIS section of `TUTORIAL_FINN_SC.pdf`).

**Different region?** Point `region.boundaries_shp` at any shapefile and set
`region.region_field` in `config/config.yml` — no code changes needed.